# Ancestría local con RFMix (cromosoma 22)

Con ADMIXTURE estimamos la **ancestría global**: qué proporción del genoma de cada persona viene de cada población ancestral. La **ancestría local** va más allá: dice de qué población viene **cada segmento** de cada cromosoma. En una persona mezclada, cada cromosoma es un mosaico de segmentos africanos, europeos y nativos americanos, porque la recombinación los ha ido cortando en cada generación desde la mezcla.

Usaremos RFMix (Maples et al., 2013) sobre el cromosoma 22 de personas de Medellín, Colombia (CLM, 1000 Genomes).

Pasos:

1. Descargar los datos
2. Revisar los cuatro insumos de RFMix
3. Instalar RFMix
4. Correr RFMix
5. Revisar los archivos de salida
6. Ancestría global de cada persona
7. Pintar los cromosomas: ancestría local
8. Comparar poblaciones con resultados precalculados

Después de cada paso hay una tabla con los archivos que crea, qué son y si se usan después.

## 1. Descargar los datos

Los datos ya están preparados en el repositorio del curso (carpeta `data/ancestria_local/`): vienen del panel HGDP + 1000 Genomes faseado (Koenig et al., 2024), solo el cromosoma 22.

In [ ]:
%%bash
URL=https://github.com/YTZ27/curso-ancestria/raw/main/data/ancestria_local
for f in chr22_consulta_clase.vcf.gz chr22_consulta_clase.vcf.gz.tbi chr22_referencia.vcf.gz chr22_referencia.vcf.gz.tbi sample_map.txt poblaciones.tsv genetic_map_chr22_GRCh38.txt rfmix_todos.msp.tsv.gz rfmix_todos.rfmix.Q; do
  wget -nv -nc $URL/$f
done
ls -lh

**Archivos de este paso**

| Archivo | Qué es | ¿Se usa después? |
|---|---|---|
| `chr22_consulta_clase.vcf.gz` | Haplotipos faseados de 10 personas de CLM: las que vamos a analizar (consulta) | Sí: paso 4 |
| `chr22_referencia.vcf.gz` | Haplotipos faseados de las poblaciones de referencia: africana (YRI), europea (IBS) y nativas americanas (HGDP) | Sí: paso 4 |
| `…vcf.gz.tbi` | Índices de los dos archivos VCF: permiten leerlos sin recorrerlos enteros | Sí: RFMix los usa si están |
| `sample_map.txt` | Para RFMix: cada persona de referencia y su ancestría (AFR, EUR o NAT) | Sí: paso 4 |
| `poblaciones.tsv` | Población, proyecto y rol (consulta o referencia) de todas las muestras | Sí: pasos 2 y 8 |
| `genetic_map_chr22_GRCh38.txt` | Mapa genético: posición en cM de cada posición del cromosoma 22 | Sí: paso 4 |
| `rfmix_todos.msp.tsv.gz` | Resultados de RFMix ya calculados para todas las personas mezcladas (CLM, MXL, PUR, PEL) | Sí: paso 8 |
| `rfmix_todos.rfmix.Q` | Ancestría global de esas mismas personas, calculada por RFMix | Sí: paso 8 |

## 2. Revisar los cuatro insumos de RFMix

RFMix necesita cuatro cosas: (1) los haplotipos faseados de las personas a analizar, (2) los haplotipos faseados de las poblaciones de referencia, (3) a qué ancestría pertenece cada referencia y (4) un mapa genético.

### 2.1 Haplotipos faseados (VCF)

In [ ]:
!zcat chr22_consulta_clase.vcf.gz | grep -v '^##' | head -4 | cut -f1-12

Fíjate en los genotipos: ahora se escriben `0|1`, con barra vertical, y no `0/1` como en el exoma. La barra vertical significa que el genotipo está **faseado**: el alelo de la izquierda está en un cromosoma de la pareja y el de la derecha en el otro. Así cada persona aporta dos **haplotipos**, y RFMix asigna ancestría a cada uno por separado.

¿Cuántos SNPs hay?

In [ ]:
!zcat chr22_consulta_clase.vcf.gz | grep -v '^#' | wc -l

### 2.2 Referencias y *sample map*

El *sample map* le dice a RFMix a qué ancestría pertenece cada persona de referencia: una línea por persona, con su identificador y la ancestría, separados por un tabulador.

In [ ]:
!head -3 sample_map.txt
!cut -f2 sample_map.txt | sort | uniq -c

Las poblaciones que hay en los datos y su rol:

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

pob = pd.read_csv('poblaciones.tsv', sep='\t')
pob.groupby(['rol', 'poblacion']).size()

**Ejercicio:** armar el *sample map* a partir de `poblaciones.tsv`. Primero decidimos qué ancestría representa cada población de referencia; luego escribimos el archivo y comprobamos que es igual al que viene con los datos.

In [ ]:
ancestria = {'YRI': 'AFR', 'IBS': 'EUR',
             'Karitiana': 'NAT', 'Surui': 'NAT', 'Pima': 'NAT', 'Colombian': 'NAT'}

ref = pob[pob['rol'] == 'referencia']
mi_map = pd.DataFrame({'muestra': ref['muestra'], 'ancestria': ref['poblacion'].map(ancestria)})
mi_map.to_csv('mi_sample_map.txt', sep='\t', index=False, header=False)

original = pd.read_csv('sample_map.txt', sep='\t', header=None, names=['muestra', 'ancestria'])
print('¿Igual al original?', mi_map.sort_values('muestra').values.tolist() == original.sort_values('muestra').values.tolist())

**Archivos de este paso**

| Archivo | Qué es | ¿Se usa después? |
|---|---|---|
| `mi_sample_map.txt` | El *sample map* armado en el ejercicio | No: es igual a `sample_map.txt`, que es el que usaremos |

### 2.3 Que los tres archivos hablen el mismo idioma

El error más común al preparar datos para RFMix es que el cromosoma se llame distinto en cada archivo: `22` en uno y `chr22` en otro. RFMix no avisa con un error claro, simplemente no encuentra variantes en común. Conviene revisarlo antes de correr nada.

In [ ]:
%%bash
echo "Consulta:    $(zcat chr22_consulta_clase.vcf.gz | grep -v '^#' | head -1 | cut -f1)"
echo "Referencia:  $(zcat chr22_referencia.vcf.gz | grep -v '^#' | head -1 | cut -f1)"
echo "Mapa:        $(head -1 genetic_map_chr22_GRCh38.txt | cut -f1)"

Los tres dicen `chr22`, así que podemos seguir. Si no coincidieran, se arregla quitando o agregando el prefijo `chr`; por ejemplo, para quitárselo al mapa genético:

```bash
sed 's/^chr//' genetic_map_chr22_GRCh38.txt > mapa_sin_chr.txt
```

### 2.4 Mapa genético

RFMix mide distancias en centimorgans (cM) y no en pares de bases, porque los segmentos de ancestría se cortan donde hay recombinación. El mapa genético da la posición en cM de cada posición del cromosoma.

In [ ]:
!head -3 genetic_map_chr22_GRCh38.txt

In [ ]:
mapa = pd.read_csv('genetic_map_chr22_GRCh38.txt', sep='\t', header=None, names=['cromosoma', 'pb', 'cM'])
ax = mapa.plot(x='pb', y='cM', legend=False, color='#6b6b6b', figsize=(7, 3.5))
ax.set(xlabel='Posición física (pb)', ylabel='Posición genética (cM)', title='Mapa genético del cromosoma 22')
for lado in ['top', 'right']:
    ax.spines[lado].set_visible(False)
plt.tight_layout()
plt.show()

Si la recombinación fuera igual en todo el cromosoma, veríamos una línea recta. Los tramos empinados son zonas de mucha recombinación y los planos, de poca. El cromosoma 22 mide unos 74 cM.

## 3. Instalar RFMix

RFMix se descarga como código fuente y se compila (tarda menos de un minuto). Su licencia permite el uso académico sin costo.

In [ ]:
!apt-get -qq install -y autoconf automake > /dev/null
!git clone -q https://github.com/slowkoni/rfmix.git
!cd rfmix && autoreconf --force --install > /dev/null 2>&1 && ./configure > /dev/null && make -s > /dev/null 2>&1
!./rfmix/rfmix 2>&1 | head -2

**Archivos de este paso**

| Archivo | Qué es | ¿Se usa después? |
|---|---|---|
| `rfmix/rfmix` | El programa RFMix | Sí: paso 4 |
| `rfmix/ (resto de la carpeta)` | El código fuente y los archivos de la compilación | No |

## 4. Correr RFMix

Las opciones obligatorias son los cuatro insumos (`-f`, `-r`, `-m`, `-g`), el nombre de los archivos de salida (`-o`) y el cromosoma (`--chromosome`).

Las otras tres son para que quepa en una clase:

| Opción | Qué hace |
|---|---|
| `--n-threads=2` | Usa los dos procesadores que da Colab |
| `-w 41` | Le da ya calculado un parámetro que RFMix normalmente calibra solo, y que depende de las referencias, no de a quién analicemos. Esa calibración tarda unos 15 minutos; el valor 41 se obtuvo corriéndola una vez con estas mismas referencias |
| `-t 50` | Usa 50 árboles en el bosque aleatorio en lugar de 100. Con estos datos, los resultados coinciden en el 99,99 % de los SNPs |

Con 10 personas tarda unos 4 minutos.

In [ ]:
!./rfmix/rfmix -f chr22_consulta_clase.vcf.gz \
       -r chr22_referencia.vcf.gz \
       -m sample_map.txt \
       -g genetic_map_chr22_GRCh38.txt \
       -o clm_chr22 \
       --chromosome=chr22 \
       --n-threads=2 \
       -w 41 \
       -t 50

### ¿Qué tan confiables son estas referencias?

Cuando RFMix hace la calibración completa (sin `-w`), antes de analizar nada se autoevalúa: arma haplotipos simulados a partir de las referencias, les borra la etiqueta y trata de reconocerlos. Imprime el resultado como **Simulation results**, una matriz donde las filas son la ancestría verdadera, las columnas la que asignó, y la diagonal son los aciertos.

Como aquí le pasamos el valor ya calculado con `-w`, RFMix se salta ese paso y esa matriz no aparece en la salida. Esta es la que salió al calibrar con estas mismas referencias:

| Ancestría verdadera | → AFR | → EUR | → NAT |
|---|---|---|---|
| **AFR** | **99,0 %** | 0,8 % | 0,2 % |
| **EUR** | 0,9 % | **98,8 %** | 0,3 % |
| **NAT** | 0,1 % | 0,4 % | **99,5 %** |

La diagonal está por encima del 98 % en las tres, es decir, las referencias se distinguen bien entre sí. Si una ancestría tuviera un porcentaje bajo, sería señal de que su referencia es poco informativa o de que está mezclada con otra, y los resultados de esa ancestría no serían confiables.

De hecho, así se decidió qué poblaciones usar como referencia nativa americana: con los Maya de HGDP incluidos, la fila NAT bajaba a 97,3 %, porque varios de ellos tienen ancestría europea y africana. Al quitarlos subió a 99,5 %.

## 5. Revisar los archivos de salida

In [ ]:
!ls -lh clm_chr22.*

**Archivos de este paso**

| Archivo | Qué es | ¿Se usa después? |
|---|---|---|
| `clm_chr22.msp.tsv` | Ancestría más probable de cada haplotipo en cada segmento del cromosoma (una fila por segmento, una columna por haplotipo; 0 = AFR, 1 = EUR, 2 = NAT) | Sí: paso 7, para pintar los cromosomas |
| `clm_chr22.rfmix.Q` | Ancestría global de cada persona: la proporción del cromosoma 22 que viene de cada ancestría | Sí: paso 6 |
| `clm_chr22.fb.tsv` | Probabilidad de cada ancestría en cada haplotipo y segmento | No en este notebook. Sirve para medir la incertidumbre y para *admixture mapping* |
| `clm_chr22.sis.tsv` | Probabilidad de que la ancestría no cambie entre un segmento y el siguiente | No en este notebook. Sirve para estudiar dónde cambia la ancestría |

La primera línea de `.msp.tsv` dice qué número corresponde a cada ancestría. RFMix las ordena alfabéticamente: AFR = 0, EUR = 1, NAT = 2.

In [ ]:
!head -3 clm_chr22.msp.tsv | cut -f1-10

## 6. Ancestría global de cada persona

RFMix también resume la ancestría global (como ADMIXTURE), pero solo con el cromosoma 22: con un solo cromosoma la estimación es menos precisa que con todo el genoma.

In [ ]:
Q = pd.read_csv('clm_chr22.rfmix.Q', sep='\t', skiprows=1).rename(columns={'#sample': 'muestra'}).set_index('muestra')
Q.round(2)

In [ ]:
colores = {'AFR': '#eb6834', 'EUR': '#2a78d6', 'NAT': '#1baf7a'}
ax = Q.sort_values('EUR')[list(colores)].plot.bar(stacked=True, width=0.9, color=colores, edgecolor='none', figsize=(8, 4))
ax.set(ylim=(0, 1), xlabel='', ylabel='Proporción de ancestría', title='CLM: ancestría global en el cromosoma 22')
ax.legend(title='Ancestría', bbox_to_anchor=(1, 1), frameon=False)
for lado in ['top', 'right']:
    ax.spines[lado].set_visible(False)
plt.tight_layout()
plt.show()

## 7. Pintar los cromosomas: ancestría local

Lo haremos de dos maneras: primero una vista compacta de barras, fácil de comparar entre personas, y después un karyograma con la forma real del cromosoma.

### 7.1 Vista compacta

Cada barra horizontal es un haplotipo (dos por persona, `.0` y `.1`), y cada color, la ancestría de ese tramo del cromosoma 22.

In [ ]:
msp = pd.read_csv('clm_chr22.msp.tsv', sep='\t', skiprows=1)
primera_linea = open('clm_chr22.msp.tsv').readline()
codigos = {int(c): a for a, c in (x.split('=') for x in primera_linea.split(':')[1].split())}
haplotipos = msp.columns[6:]
print(codigos, '|', len(msp), 'segmentos,', len(haplotipos), 'haplotipos')

In [ ]:
fig, ax = plt.subplots(figsize=(11, 0.35 * len(haplotipos) + 1))
for fila, h in enumerate(haplotipos):
    for codigo, nombre in codigos.items():
        tramo = msp[msp[h] == codigo]
        ax.broken_barh(list(zip(tramo['spos'] / 1e6, (tramo['epos'] - tramo['spos']) / 1e6)),
                       (fila - 0.4, 0.8), color=colores[nombre])
ax.set_yticks(range(len(haplotipos)))
ax.set_yticklabels(haplotipos, fontsize=8)
ax.invert_yaxis()
ax.set(xlabel='Posición en el cromosoma 22 (Mb)', title='Ancestría local en el cromosoma 22 (CLM)')
ax.legend(handles=[plt.Rectangle((0, 0), 1, 1, color=c) for c in colores.values()], labels=list(colores),
          title='Ancestría', bbox_to_anchor=(1, 1), frameon=False)
for lado in ['top', 'right', 'left']:
    ax.spines[lado].set_visible(False)
plt.tight_layout()
plt.show()

**Cómo leerlo:** los dos haplotipos de una misma persona son distintos porque vienen de su madre y de su padre. Los segmentos largos indican mezcla más reciente: cada generación de recombinación los acorta. La proporción de cada color en los dos haplotipos de una persona es, aproximadamente, su ancestría global del paso 6.

### 7.2 Karyograma

La misma información, pero dibujando el cromosoma con su forma real: las dos copias de cada persona, una al lado de la otra, con el estrechamiento del centrómero. Es la forma clásica de presentar ancestría local en los artículos.

Para dibujarlo necesitamos tres datos del cromosoma 22 en GRCh38: su largo (50 818 468 pb) y dónde empieza y termina el centrómero (13 700 001 a 17 400 000 pb), según las bandas citogenéticas del ensamblaje.

In [ ]:
import numpy as np
from matplotlib.patches import Polygon

LARGO = 50818468                         # largo del cromosoma 22 (GRCh38)
CEN_INI, CEN_FIN = 13700001, 17400000    # centrómero
ANCHO = 0.30                             # ancho de cada cromosoma en el gráfico


def silueta(x):
    """Contorno del cromosoma: puntas redondeadas y estrechamiento en el centrómero."""
    r, tapa, cen = ANCHO / 2, 0.9e6, (CEN_INI + CEN_FIN) / 2
    izq = [(-r * np.sin(t), tapa * (1 - np.cos(t))) for t in np.linspace(0, np.pi / 2, 14)]
    izq += [(-r, CEN_INI), (-r * 0.5, cen), (-r, CEN_FIN)]
    izq += [(-r * np.sin(t), LARGO - tapa * (1 - np.cos(t))) for t in np.linspace(np.pi / 2, 0, 14)]
    der = [(-px, py) for px, py in reversed(izq)]
    return [(x + px, py) for px, py in izq + der]


def pintar(ax, columna, x):
    """Dibuja un haplotipo: dentro de la silueta, un rectángulo por segmento."""
    cuerpo = Polygon(silueta(x), closed=True, facecolor='#f2f2f0', edgecolor='none', zorder=2)
    ax.add_patch(cuerpo)
    for _, fila in msp.iterrows():
        rect = plt.Rectangle((x - ANCHO / 2, fila['spos']), ANCHO, fila['epos'] - fila['spos'],
                             facecolor=colores[codigos[fila[columna]]], edgecolor='none', zorder=3)
        ax.add_patch(rect)
        rect.set_clip_path(cuerpo)   # recorta la pintura al contorno del cromosoma
    ax.add_patch(Polygon(silueta(x), closed=True, fill=False, edgecolor='#444444', linewidth=0.9, zorder=4))

In [ ]:
muestras = sorted({c.rsplit('.', 1)[0] for c in haplotipos})

fig, ax = plt.subplots(figsize=(1.0 * len(muestras) + 2.4, 7.5))
for i, m in enumerate(muestras):
    pintar(ax, f'{m}.0', i - 0.19)
    pintar(ax, f'{m}.1', i + 0.19)

ax.set_xticks(range(len(muestras)))
ax.set_xticklabels(muestras, rotation=45, ha='right', fontsize=9)
ax.set_xlim(-0.7, len(muestras) - 0.3)
ax.set_ylim(LARGO, 0)                       # el cromosoma se dibuja de arriba hacia abajo
ax.set_yticks(np.arange(0, 51e6, 10e6))
ax.set_yticklabels([int(v / 1e6) for v in np.arange(0, 51e6, 10e6)])
ax.set(ylabel='Posición en el cromosoma 22 (Mb)', title='CLM: ancestría local en el cromosoma 22')
ax.tick_params(axis='x', length=0)
for lado in ['top', 'right', 'bottom']:
    ax.spines[lado].set_visible(False)
ax.legend(handles=[plt.Rectangle((0, 0), 1, 1, color=c) for c in colores.values()], labels=list(colores),
          title='Ancestría', bbox_to_anchor=(1, 1), frameon=False)
plt.tight_layout()
plt.show()

**La parte gris de arriba** es el brazo corto del cromosoma 22. Es un cromosoma acrocéntrico: ese brazo es muy pequeño y está formado por secuencia repetitiva, así que no tiene SNPs utilizables y RFMix no le asigna ancestría.

Este estilo de figura viene de **AncestryGrapher** (Lisi y Campbell, Universidad del Sur de California; [alisi1989/RFMIX2-Pipeline-to-plot](https://github.com/alisi1989/RFMIX2-Pipeline-to-plot)), un conjunto de herramientas que convierte las salidas de RFMix en figuras publicables: **GAP** para ancestría global y **LAP** para pintar los tractos de ancestría sobre el karyograma completo de los 22 cromosomas.

Aquí lo reprodujimos solo para el cromosoma 22 y con matplotlib, para no depender de instalaciones adicionales en Colab. Si corres RFMix sobre todo el genoma, esa herramienta es el camino, y sus autores piden que se la cite.

Los pasos 7 y 8 no crean archivos: los gráficos se muestran aquí.

## 8. Comparar poblaciones con resultados precalculados

Correr RFMix con las 345 personas mezcladas de los datos (CLM, MXL, PUR y PEL) tarda demasiado para la clase, así que esos resultados ya vienen calculados (`rfmix_todos.*`), con los mismos insumos y opciones del paso 4.

Primero, la ancestría global promedio de cada población:

In [ ]:
Q_todos = pd.read_csv('rfmix_todos.rfmix.Q', sep='\t', skiprows=1).rename(columns={'#sample': 'muestra'})
Q_todos = Q_todos.merge(pob[['muestra', 'poblacion']], on='muestra')
Q_todos.groupby('poblacion')[list(colores)].mean().round(2)

In [ ]:
promedio = Q_todos.groupby('poblacion')[list(colores)].mean().loc[['PUR', 'CLM', 'MXL', 'PEL']]
ax = promedio.plot.bar(stacked=True, color=colores, edgecolor='none', width=0.7, figsize=(6, 4))
ax.set(ylim=(0, 1), xlabel='', ylabel='Proporción de ancestría', title='Ancestría global promedio (cromosoma 22)')
ax.tick_params(axis='x', rotation=0)
ax.legend(title='Ancestría', bbox_to_anchor=(1, 1), frameon=False)
for lado in ['top', 'right']:
    ax.spines[lado].set_visible(False)
plt.tight_layout()
plt.show()

Y la ancestría local a lo largo del cromosoma: en cada segmento, qué fracción de los haplotipos de cada población tiene cada ancestría.

In [ ]:
msp_todos = pd.read_csv('rfmix_todos.msp.tsv.gz', sep='\t', skiprows=1)
haplo_pob = pd.Series({h: h.rsplit('.', 1)[0] for h in msp_todos.columns[6:]}).map(pob.set_index('muestra')['poblacion'])
centro = (msp_todos['spos'] + msp_todos['epos']) / 2 / 1e6

fig, axes = plt.subplots(4, 1, figsize=(10, 9), sharex=True, sharey=True)
for ax, p in zip(axes, ['PUR', 'CLM', 'MXL', 'PEL']):
    columnas = haplo_pob[haplo_pob == p].index
    for codigo, nombre in codigos.items():
        ax.plot(centro, (msp_todos[columnas] == codigo).mean(axis=1), color=colores[nombre], label=nombre, linewidth=1.5)
    ax.set_title(p, loc='left')
    ax.set_ylim(0, 1)
    for lado in ['top', 'right']:
        ax.spines[lado].set_visible(False)
axes[0].legend(title='Ancestría', bbox_to_anchor=(1, 1), frameon=False)
axes[-1].set_xlabel('Posición en el cromosoma 22 (Mb)')
fig.supylabel('Fracción de haplotipos')
plt.tight_layout()
plt.show()

**Cómo leerlo:** si la mezcla fuera uniforme, cada línea sería casi plana y estaría a la altura de la ancestría global de la población. Las desviaciones grandes y sostenidas en una región pueden deberse al azar, a errores, o a selección natural después de la mezcla; con varias poblaciones y todo el genoma, esas regiones se estudian con *admixture mapping*.

## Para la próxima sesión

Con el archivo `.msp.tsv` se puede preguntar por una posición concreta del genoma: por ejemplo, qué ancestría tienen los dos haplotipos de una persona justo donde está una variante asociada a una enfermedad. Esa es la base del *admixture mapping* y de la parte de aplicaciones de la sesión 6.

## Referencias

- Maples BK, Gravel S, Kenny EE, Bustamante CD. RFMix: a discriminative modeling approach for rapid and robust local-ancestry inference. *Am J Hum Genet*. 2013;93:278–288. doi:10.1016/j.ajhg.2013.06.020
- Koenig Z, et al. A harmonized public resource of deeply sequenced diverse human genomes. *Genome Res*. 2024;34(5):796. doi:10.1101/gr.278378.123
- Gravel S. Population genetics models of local ancestry. *Genetics*. 2012;191:607–619. doi:10.1534/genetics.112.139808 (longitud de los segmentos y tiempo desde la mezcla)